<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/analisis/nb6_crtbp_jacobi_tisserand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6 — CRTBP, Constante de Jacobi y Parámetro de Tisserand
## El sistema Sol–Tierra–Apophis como problema de tres cuerpos

**Autor:** Soleil Dayana Niño Murcia  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Explorar la dinámica de Apophis dentro del marco del CRTBP Sol-Tierra:
> (A) calcular la constante de Jacobi y las superficies de velocidad cero para determinar si
> Apophis puede ser capturado por la Tierra, y (B) calcular el parámetro de Tisserand para
> demostrar su conservación antes y después del encuentro de 2029.

## Bloque A — CRTBP y Constante de Jacobi

### A.1 Teoría: El Problema Circular Restringido de Tres Cuerpos

El **CRTBP** describe el movimiento de una partícula de masa despreciable (Apophis) en el campo
gravitacional de dos cuerpos primarios que orbitan circularmente (Sol y Tierra).

#### Sistema de referencia rotante

Se adopta un sistema que rota con la frecuencia orbital de la Tierra ($n = 2\pi/T$), donde:
- El **Sol** se ubica en $(-\mu, 0)$ y la **Tierra** en $(1-\mu, 0)$
- El parámetro de masa: $\mu = M_\oplus / (M_\odot + M_\oplus) \approx 3 \times 10^{-6}$
- La distancia Sol-Tierra se normaliza a 1

#### Potencial efectivo

$$\Omega(x, y) = \frac{1-\mu}{r_1} + \frac{\mu}{r_2} + \frac{x^2 + y^2}{2}$$

donde $r_1 = \sqrt{(x+\mu)^2 + y^2}$ (distancia al Sol) y $r_2 = \sqrt{(x-1+\mu)^2 + y^2}$ (a la Tierra).

#### Constante de Jacobi

La **única cuadratura** del CRTBP es la constante de Jacobi:

$$\boxed{C_J = 2\Omega(x,y) - v^2}$$

donde $v^2 = \dot{x}^2 + \dot{y}^2$ es la velocidad en el marco rotante. $C_J$ se conserva a lo largo
de cualquier trayectoria no perturbada.

#### Superficies de velocidad cero

Para $v = 0$: $C_J = 2\Omega(x,y)$. Estas curvas delimitan regiones prohibidas al movimiento.
Los **cinco puntos de Lagrange** son los extremos del potencial efectivo $\Omega$, y sus valores
$C_J(L_i)$ son umbrales críticos:
- Si $C_J^{\text{Apophis}} > C_J(L_1)$: la puerta $L_1$ está **cerrada** (captura imposible)
- Si $C_J^{\text{Apophis}} < C_J(L_1)$: la puerta $L_1$ está **abierta**

In [ ]:
%pip install -Uq pymcel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from scipy.optimize import fsolve, brentq

import pymcel as pc
from pymcel import constantes as const

print('Librerías cargadas correctamente.')

In [ ]:
# ── Parámetro de masa del sistema Sol-Tierra ─────────────────────────────
AU_km    = 149_597_870.7
AU_m     = AU_km * 1e3
M_sun_kg = 1.989e30
G_SI     = 6.674e-11
UT_s     = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))
UT_days  = UT_s / 86_400.0
vel_unit = AU_km / UT_s

GM_sun_km3s2   = 1.32712440018e11
GM_earth_km3s2 = 3.986004418e5

# Parámetro mu del CRTBP Sol-Tierra
MU = GM_earth_km3s2 / (GM_sun_km3s2 + GM_earth_km3s2)
print(f'μ (CRTBP Sol-Tierra) = {MU:.6e}')
print(f'UT = {UT_days:.4f} días  |  1 AU/UT = {vel_unit:.4f} km/s')

### A.2 Potencial efectivo y constante de Jacobi

In [ ]:
def omega_efectivo(x, y, mu):
    """Potencial efectivo del CRTBP en 2D."""
    r1 = np.sqrt((x + mu)**2 + y**2)          # distancia al Sol (en -mu, 0)
    r2 = np.sqrt((x - 1 + mu)**2 + y**2)      # distancia a la Tierra (en 1-mu, 0)
    return (1 - mu) / r1 + mu / r2 + (x**2 + y**2) / 2

def jacobi_constant(x, y, vx, vy, mu):
    """Constante de Jacobi C_J = 2*Omega - v^2."""
    v2 = vx**2 + vy**2
    return 2 * omega_efectivo(x, y, mu) - v2

# Función para encontrar los puntos colineales de Lagrange (L1, L2, L3)
def gradOmega_x_colineal(x, mu):
    """dΩ/dx = 0 en el eje y=0 (puntos colineales)."""
    r1 = abs(x + mu)
    r2 = abs(x - 1 + mu)
    s1 = np.sign(x + mu)
    s2 = np.sign(x - 1 + mu)
    return x - (1 - mu) * s1 / r1**2 - mu * s2 / r2**2

# Localizar L1, L2, L3 numéricamente
x_L1 = brentq(gradOmega_x_colineal, 1 - MU - 0.1, 1 - MU - 1e-6, args=(MU,))
x_L2 = brentq(gradOmega_x_colineal, 1 - MU + 1e-6, 1 - MU + 0.1, args=(MU,))
x_L3 = brentq(gradOmega_x_colineal, -1.5, -MU - 1e-6, args=(MU,))

# L4, L5 están en (0.5 - mu, ±sqrt(3)/2)
x_L4, y_L4 = 0.5 - MU,  np.sqrt(3) / 2
x_L5, y_L5 = 0.5 - MU, -np.sqrt(3) / 2

# Valores de C_J en cada punto de Lagrange
lagrange_pts = {
    'L1': (x_L1, 0), 'L2': (x_L2, 0), 'L3': (x_L3, 0),
    'L4': (x_L4, y_L4), 'L5': (x_L5, y_L5),
}

print('Puntos de Lagrange y sus constantes de Jacobi (C_J = 2Ω):')
print(f'{"Punto":<5} {"x":>10} {"y":>10} {"C_J":>12}')
print('-' * 42)
CJ_lagrange = {}
for nombre, (xl, yl) in lagrange_pts.items():
    CJ_l = 2 * omega_efectivo(xl, yl, MU)
    CJ_lagrange[nombre] = CJ_l
    print(f'{nombre:<5} {xl:>10.6f} {yl:>10.6f} {CJ_l:>12.6f}')

### A.3 Mapa de Curvas de Velocidad Cero del sistema Sol-Tierra

In [ ]:
# Grid del plano rotante
NG = 400
rango = 1.5
x_grid = np.linspace(-rango, rango, NG)
y_grid = np.linspace(-rango, rango, NG)
X, Y = np.meshgrid(x_grid, y_grid)

# C_J en cada punto con v=0
CJ_map = 2 * omega_efectivo(X, Y, MU)

# Saturar valores muy altos (cerca de los primarios)
CJ_plot = np.clip(CJ_map, 2.5, 5.0)

fig, ax = plt.subplots(figsize=(9, 8))

cf = ax.contourf(X, Y, CJ_plot,
                  levels=np.linspace(2.5, 5.0, 30),
                  cmap='RdYlBu_r', alpha=0.85)
plt.colorbar(cf, ax=ax, label='$C_J = 2\\Omega(x,y)$  (velocidad cero)', shrink=0.8)

# Curvas de nivel en los valores críticos de los puntos de Lagrange
nivel_colores = {'L1': 'red', 'L2': 'darkorange', 'L3': 'darkgreen'}
for nombre in ['L1', 'L2', 'L3']:
    cj = CJ_lagrange[nombre]
    if 2.5 <= cj <= 5.0:
        ax.contour(X, Y, CJ_map, levels=[cj],
                   colors=nivel_colores[nombre], linewidths=1.5, linestyles='--')

# Marcar los cuerpos principales
ax.plot(-MU, 0, 'yo', markersize=12, label='Sol', zorder=5)
ax.plot(1 - MU, 0, 'bo', markersize=7, label='Tierra', zorder=5)

# Marcar puntos de Lagrange
marcadores = {'L1': 'r^', 'L2': 'g^', 'L3': 'm^', 'L4': 'c^', 'L5': 'c^'}
for nombre, (xl, yl) in lagrange_pts.items():
    ax.plot(xl, yl, marcadores[nombre], markersize=8, zorder=5)
    ax.annotate(nombre, (xl, yl), textcoords='offset points',
                xytext=(5, 5), fontsize=9, fontweight='bold')

ax.set_xlim(-rango, rango)
ax.set_ylim(-rango, rango)
ax.set_xlabel('$x$ (unidades Sol-Tierra)', fontsize=11)
ax.set_ylabel('$y$ (unidades Sol-Tierra)', fontsize=11)
ax.set_title('Curvas de velocidad cero — CRTBP Sol-Tierra\n'
             'Líneas de trazos: niveles $C_J(L_1)$, $C_J(L_2)$, $C_J(L_3)$',
             fontsize=11)
ax.legend(fontsize=9, loc='upper right')
ax.set_aspect('equal')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('crtbp_curvas_velocidad_cero.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: crtbp_curvas_velocidad_cero.png')

### A.4 Constante de Jacobi de Apophis: ¿puede ser capturado?

In [ ]:
# Obtener posición y velocidad de Apophis en el marco inercial
EPOCA_ANALISIS = '2029-04-13'  # día del acercamiento

print(f'Consultando estado de Apophis en {EPOCA_ANALISIS}...')
tabla_apo, jd_apo, estado_apo = pc.consulta_horizons(
    id='99942', location='@0', epochs=EPOCA_ANALISIS
)
r_apo = estado_apo[:3]              # AU
v_apo = estado_apo[3:] * UT_days   # AU/UT

print(f'Consultando estado de la Tierra en {EPOCA_ANALISIS}...')
tabla_tie, jd_tie, estado_tie = pc.consulta_horizons(
    id='399', location='@0', epochs=EPOCA_ANALISIS
)
r_tie = estado_tie[:3]
v_tie = estado_tie[3:] * UT_days

print(f'Consultando estado del Sol en {EPOCA_ANALISIS}...')
tabla_sol, jd_sol, estado_sol = pc.consulta_horizons(
    id='10', location='@0', epochs=EPOCA_ANALISIS
)
r_sol = estado_sol[:3]
v_sol = estado_sol[3:] * UT_days

print('OK')

In [ ]:
# Conversión al marco rotante Sol-Tierra
# ─────────────────────────────────────────────────────────────────────────
# Vector Sol→Tierra en el inercial
r_ST = r_tie - r_sol         # vector de Sol a Tierra
d_ST = np.linalg.norm(r_ST)  # distancia Sol-Tierra

# Eje x_rot apunta de Sol a Tierra
x_hat = r_ST / d_ST
# Eje z_rot = h_ST / |h_ST|
h_ST = np.cross(r_ST, v_tie - v_sol)
z_hat = h_ST / np.linalg.norm(h_ST)
y_hat = np.cross(z_hat, x_hat)

# Velocidad angular del sistema Sol-Tierra
omega_ST = np.linalg.norm(h_ST) / d_ST**2  # rad/UT

# Posición de Apophis en el marco rotante (normalizada a d_ST = 1)
r_apo_rel = (r_apo - r_sol) / d_ST
x_rot = np.dot(r_apo_rel, x_hat)
y_rot = np.dot(r_apo_rel, y_hat)

# Velocidad en el marco rotante
v_apo_rel = (v_apo - v_sol) / (d_ST * omega_ST)
vx_inercial = np.dot(v_apo_rel, x_hat)
vy_inercial = np.dot(v_apo_rel, y_hat)
# Corregir por rotación del marco: v_rot = v_ine - ω × r_rot
vx_rot = vx_inercial + y_rot   # en unidades CRTBP, omega = 1
vy_rot = vy_inercial - x_rot

print(f'Posición de Apophis en marco rotante (2029-04-13):')
print(f'  x_rot = {x_rot:.6f} u.S-T  |  y_rot = {y_rot:.6f} u.S-T')
print(f'Velocidad en marco rotante:')
print(f'  vx_rot = {vx_rot:.6f}  |  vy_rot = {vy_rot:.6f}')

# Constante de Jacobi de Apophis
CJ_apophis = jacobi_constant(x_rot, y_rot, vx_rot, vy_rot, MU)
print(f'\nConstante de Jacobi de Apophis:')
print(f'  C_J(Apophis) = {CJ_apophis:.6f}')
print(f'  C_J(L1)      = {CJ_lagrange["L1"]:.6f}')
print(f'  C_J(L2)      = {CJ_lagrange["L2"]:.6f}')
print()

# Diagnóstico: ¿puerta abierta o cerrada?
if CJ_apophis > CJ_lagrange['L1']:
    print('► Puerta L1: CERRADA  (C_J > C_J(L1)) → Apophis NO puede cruzar L1')
else:
    print('► Puerta L1: ABIERTA  (C_J < C_J(L1)) → Apophis SÍ puede cruzar L1')

if CJ_apophis > CJ_lagrange['L2']:
    print('► Puerta L2: CERRADA  (C_J > C_J(L2)) → Apophis NO puede cruzar L2')
else:
    print('► Puerta L2: ABIERTA  (C_J < C_J(L2)) → Apophis SÍ puede cruzar L2')

print()
if CJ_apophis < CJ_lagrange['L2']:
    print('✓ CONCLUSIÓN: Las superficies de velocidad cero NO restringen a Apophis.')
    print('  Apophis tiene suficiente energía para escapar del sistema Tierra.')
else:
    print('! CONCLUSIÓN: Apophis podría quedar temporalmente atrapado en la región terrestre.')

In [ ]:
# Mapa C_J con la posición de Apophis superpuesta
fig, ax = plt.subplots(figsize=(9, 8))

# Zoom en la región de la Tierra
rango_zoom = 0.05  # unidades Sol-Tierra
x_zoom = np.linspace(1 - MU - rango_zoom, 1 - MU + rango_zoom, 300)
y_zoom = np.linspace(-rango_zoom, rango_zoom, 300)
Xz, Yz = np.meshgrid(x_zoom, y_zoom)
CJ_zoom = 2 * omega_efectivo(Xz, Yz, MU)
CJ_zoom_plot = np.clip(CJ_zoom, 3.0, 5.0)

cf = ax.contourf(Xz, Yz, CJ_zoom_plot,
                  levels=np.linspace(3.0, 5.0, 30),
                  cmap='RdYlBu_r', alpha=0.85)
plt.colorbar(cf, ax=ax, label='$C_J = 2\\Omega(x,y)$', shrink=0.8)

# Curvas de nivel L1, L2
for nombre, color_l in [('L1', 'red'), ('L2', 'orange')]:
    cj = CJ_lagrange[nombre]
    if cj >= 3.0:
        cs = ax.contour(Xz, Yz, CJ_zoom, levels=[cj],
                        colors=color_l, linewidths=2, linestyles='--')
        ax.clabel(cs, fmt=f'$C_J({nombre})$={{:.3f}}'.format(cj), fontsize=8)

# Tierra y L1, L2
ax.plot(1 - MU, 0, 'b^', markersize=10, label='Tierra', zorder=5)
ax.plot(x_L1, 0, 'r^', markersize=8, label=f'L1 (x={x_L1:.5f})', zorder=5)
ax.plot(x_L2, 0, 'g^', markersize=8, label=f'L2 (x={x_L2:.5f})', zorder=5)

# Posición de Apophis
ax.plot(x_rot, y_rot, 'k*', markersize=14, label=f'Apophis ($C_J$={CJ_apophis:.4f})', zorder=6)

# Línea horizontal de nivel de Apophis
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

ax.set_xlim(1 - MU - rango_zoom, 1 - MU + rango_zoom)
ax.set_ylim(-rango_zoom, rango_zoom)
ax.set_xlabel('$x$ (unidades Sol-Tierra)', fontsize=11)
ax.set_ylabel('$y$ (unidades Sol-Tierra)', fontsize=11)
ax.set_title(f'Zoom región Tierra — Apophis (2029-04-13)\n'
             f'$C_J$(Apophis)={CJ_apophis:.4f}  vs.  $C_J(L_1)$={CJ_lagrange["L1"]:.4f}',
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('crtbp_apophis_jacobi.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: crtbp_apophis_jacobi.png')

---
## Bloque B — Parámetro de Tisserand

### B.1 Teoría

El **parámetro de Tisserand** es una aproximación de la constante de Jacobi que se puede calcular
solo con los elementos orbitales clásicos:

$$\boxed{T_P = \frac{a_P}{a} + 2\cos i \sqrt{\frac{a}{a_P}(1-e^2)}}$$

donde:
- $a_P$ = semieje mayor del **cuerpo perturbador** (aquí la Tierra: $a_P = 1$ AU)
- $a$, $e$, $i$ = elementos orbitales del objeto de interés (Apophis)
- $i$ = inclinación respecto al plano orbital del perturbador

#### Propiedades clave:

1. $T_P$ es una **integral aproximada** del CRTBP
2. Se **conserva a lo largo de encuentros cercanos** con el perturbador (Tierra)
3. Es la "prueba de identidad" de los cometas/asteroides: aunque $a$, $e$, $i$ cambien
   drásticamente, $T_P$ permanece casi constante
4. Para objetos ligados a la Tierra: $T_P < 3$; asteroides del cinturón: $T_P > 3$

In [ ]:
def tisserand(a, e, i_rad, a_perturb=1.0):
    """Parámetro de Tisserand respecto al perturbador con semieje a_perturb (AU)."""
    return a_perturb / a + 2 * np.cos(i_rad) * np.sqrt((a / a_perturb) * (1 - e**2))

# Test rápido
TP_test = tisserand(a=0.92, e=0.191, i_rad=np.radians(3.34))
print(f'Test: T_P(Apophis aprox.) = {TP_test:.4f}')

### B.2 Evolución temporal del parámetro de Tisserand de Apophis (2027–2031)

In [ ]:
# Consultar elementos orbitales de Apophis en una grilla de épocas
print('Descargando elementos orbitales de Apophis 2027-2031...')

epochs_tisserand = {
    'start': '2027-01-01',
    'stop':  '2031-01-01',
    'step':  '30d'
}

mu_canon = 1.0

tabla_ts, jd_ts, estados_ts = pc.consulta_horizons(
    id       = '99942',
    location = '@0',
    epochs   = epochs_tisserand,
)
print(f'  {len(jd_ts)} épocas descargadas.')

# Calcular elementos orbitales y T_P para cada época
n_epocas = len(jd_ts)
TP_arr   = np.zeros(n_epocas)
a_arr    = np.zeros(n_epocas)
e_arr    = np.zeros(n_epocas)
i_arr    = np.zeros(n_epocas)

for k in range(n_epocas):
    r_k = estados_ts[k, :3]
    v_k = estados_ts[k, 3:] * UT_days
    estado_k = np.concatenate([r_k, v_k])
    try:
        elems_k = pc.estado_a_elementos(mu_canon, estado_k)
        p_k, e_k, i_k = elems_k[0], elems_k[1], elems_k[2]
        a_k = p_k / (1 - e_k**2)
        TP_arr[k] = tisserand(a_k, e_k, i_k, a_perturb=1.0)
        a_arr[k]  = a_k
        e_arr[k]  = e_k
        i_arr[k]  = np.degrees(i_k)
    except Exception as ex:
        TP_arr[k] = np.nan
        print(f'  Advertencia en k={k}: {ex}')

# Fechas en días desde J2000
jd_ref   = 2451545.0  # J2000
t_dias   = np.array(jd_ts) - jd_ref
t_years  = t_dias / 365.25 + 2000.0

# Fecha del encuentro (2029-04-13 ≈ JD 2462899)
jd_encuentro = 2462899.5
t_encuentro  = (jd_encuentro - jd_ref) / 365.25 + 2000.0

print(f'T_P calculado para {np.sum(~np.isnan(TP_arr))} épocas.')
print(f'T_P medio PRE  (t < encuentro): {np.nanmean(TP_arr[t_years < t_encuentro]):.6f}')
print(f'T_P medio POST (t > encuentro): {np.nanmean(TP_arr[t_years > t_encuentro]):.6f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

# Panel 1: Parámetro de Tisserand
axes[0].plot(t_years, TP_arr, 'b-', linewidth=1.5, label='$T_P$ (Apophis–Tierra)')
axes[0].axvline(t_encuentro, color='red', linestyle='--', linewidth=2, label='Encuentro 2029')
axes[0].axhline(3.0, color='gray', linestyle=':', linewidth=1, label='$T_P = 3$ (frontera)')

# Banda de ±0.5% alrededor de la media
TP_pre_mean = np.nanmean(TP_arr[t_years < t_encuentro])
axes[0].axhspan(TP_pre_mean * 0.995, TP_pre_mean * 1.005,
                color='blue', alpha=0.1, label='±0.5% de la media')
axes[0].set_ylabel('$T_P$', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_title('Parámetro de Tisserand de Apophis respecto a la Tierra (2027–2031)', fontsize=11)

# Panel 2: semieje mayor
axes[1].plot(t_years, a_arr, 'g-', linewidth=1.5)
axes[1].axvline(t_encuentro, color='red', linestyle='--', linewidth=2)
axes[1].set_ylabel('$a$ (AU)', fontsize=11)
axes[1].grid(alpha=0.3)

# Panel 3: excentricidad
axes[2].plot(t_years, e_arr, 'm-', linewidth=1.5)
axes[2].axvline(t_encuentro, color='red', linestyle='--', linewidth=2)
axes[2].set_ylabel('$e$', fontsize=11)
axes[2].set_xlabel('Año', fontsize=11)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tisserand_apophis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: tisserand_apophis.png')

### B.3 Análisis de conservación

In [ ]:
mask_pre  = (t_years < t_encuentro) & ~np.isnan(TP_arr)
mask_post = (t_years > t_encuentro) & ~np.isnan(TP_arr)

TP_pre_mean  = TP_arr[mask_pre].mean()
TP_post_mean = TP_arr[mask_post].mean()
TP_pre_std   = TP_arr[mask_pre].std()
TP_post_std  = TP_arr[mask_post].std()
delta_TP_pct = 100 * abs(TP_post_mean - TP_pre_mean) / TP_pre_mean

a_pre_mean  = a_arr[mask_pre].mean()
a_post_mean = a_arr[mask_post].mean()
delta_a_pct = 100 * abs(a_post_mean - a_pre_mean) / a_pre_mean

e_pre_mean  = e_arr[mask_pre].mean()
e_post_mean = e_arr[mask_post].mean()
delta_e_pct = 100 * abs(e_post_mean - e_pre_mean) / e_pre_mean

print('=' * 60)
print('Análisis de conservación del Parámetro de Tisserand')
print('=' * 60)
print(f'  T_P pre-encuentro  : {TP_pre_mean:.6f}  ± {TP_pre_std:.2e}')
print(f'  T_P post-encuentro : {TP_post_mean:.6f}  ± {TP_post_std:.2e}')
print(f'  ΔT_P/T_P           : {delta_TP_pct:.4f}%')
print('-' * 60)
print(f'  Δa/a  (semieje)    : {delta_a_pct:.4f}%  (puede cambiar significativamente)')
print(f'  Δe/e  (excentr.)   : {delta_e_pct:.4f}%  (puede cambiar significativamente)')
print('=' * 60)

if delta_TP_pct < 1.0:
    print(f'\n✓ ΔT_P/T_P < 1% → Parámetro de Tisserand CONSERVADO')
    print('  Aunque a y e cambian, T_P permanece casi constante.')
    print('  Esto confirma la utilidad de T_P como "huella digital" orbital.')
else:
    print(f'\n! ΔT_P/T_P = {delta_TP_pct:.2f}% > 1%')
    print('  La perturbación es suficientemente fuerte para modificar T_P.')

print('\nValidación cruzada con C_J:')
print(f'  C_J(Apophis) en el encuentro = {CJ_apophis:.6f}')
print(f'  T_P(Apophis) PRE             = {TP_pre_mean:.6f}')
print(f'  Diferencia C_J vs T_P        = {abs(CJ_apophis - TP_pre_mean):.4f}')
print('  (T_P ≈ C_J es la aproximación esperada para μ → 0)')